# ДЗ 4. Операции, axis и broadcasting на данных «Дала Маркет»
Срок: **до понедельника 16.11, 19:00**.
Сдача: папка `hw04` репозитория `dsa-homework` (`hw04.ipynb`): `git add hw04` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число + одна фраза для менеджера**. Циклы `for` по заказам не используем. Небольшой цикл по городам или категориям при сборке таблицы допустим.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import numpy as np
# 1
a = np.array([2, 4, 6]); print(a / 2, a // 4)
# 2
m = np.ones((3, 4)); print(m.sum(axis=1))
# 3
print(np.arange(3) + np.arange(3).reshape(3, 1))
# 4
x = np.array([1, 4, 9]); print(np.sqrt(x).sum())
# 5
print((np.arange(1, 5) ** 2).mean())
```

Ваши ответы и объяснения:

1. 
2. 
3. 
4. 
5. 

In [86]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [88]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])
month  = np.array([r["order_date"][:7] for r in rows])      # "2026-09" или "2026-10"
cat    = np.array([r["category"] for r in rows])
oid    = np.array([r["order_id"] for r in rows])

print(amount.shape, np.unique(month))

(5463,) ['2026-09' '2026-10']


In [53]:
# Чистые данные: без дублей (оставляем первое вхождение order_id), без TEST, только оплаченные
_, first = np.unique(oid, return_index=True)
keep = np.zeros(len(oid), dtype=bool)
keep[first] = True
clean = keep & (cust != "TEST") & paid
print("строк всего:", len(oid), "| в чистых данных:", clean.sum())
# Допонительные обязательные обработки
cities = np.unique(city)
categories = np.unique(cat)


строк всего: 5463 | в чистых данных: 4739


## [База] 1. Выручка по чистым данным
Сколько строк осталось в `clean`? Выручка за сентябрь и октябрь и изменение в %. Совпадает ли с ДЗ 2?

In [17]:
# 1
months = np.array(["2026-09", "2026-10"])

table = np.array([amount[clean & (month == m)].sum() for m in months])
prev_month_name = None
prev_revenue = 0
for c, r in zip(months, table):
    lsum=r
    if prev_month_name is None:
        change_str = "---"
    else:
        if prev_revenue > 0:
            pct_change = ((lsum - prev_revenue) / prev_revenue) * 100
            change_str = f"{pct_change:+.2f}%"
        else:
            change_str = "N/A (в пред. месяце 0)"
    prev_month_name = c
    prev_revenue = lsum

    print(f"{c:<10} | {r:>12,.0f} ₸  | {change_str}")
print("Все совпадает")

2026-09    |   35,243,410 ₸  | ---
2026-10    |   34,419,520 ₸  | -2.34%


## [База] 2. Таблица «категория × месяц»
Соберите таблицу 5 × 2 по образцу таблицы городов с урока (категории: `np.unique(cat)`).
Какая категория приносит больше всего выручки за два месяца (`axis`)? Какая упала сильнее всех в %?

In [45]:
# 2

months = np.array(["2026-09", "2026-10"])

# Таблица 6 × 2: строка = город, столбец = месяц
table = np.array([[amount[clean & (cat == c) & (month == m)].sum() for m in months]
                  for c in categories])
for c, r in zip(categories, table):
    print(f"{c:<10} {r[0]:>12,.0f} {r[1]:>12,.0f}")
print(table.shape)
catMax=table.sum(axis=1)
catMaxIndex=catMax.argmax()
print(f"Категория с максимальной выручкой {categories[catMaxIndex]:<10} {catMax[catMaxIndex]:>12,.0f} ")

x=table[:, 1] / table[:, 0] - 1
catMinIndex=x.argmin()
# check=table.sum(axis=1).argmax()
print(f"Категория с максимальным падением {categories[catMinIndex]:<10} {x[catMinIndex]:>+12.1%}")

for c, r in zip(categories, x):
    print(f"{c:<10} {r:>+12.1%}")
# # for c in categories:
# #     print(f"{c:<10} {r[0]:>12,.0f} {r[1]:>12,.0f}")
# print(table.shape)

Дом           4,997,730    5,125,790
Книги         1,065,300    1,140,910
Красота       2,610,950    2,678,990
Одежда        9,167,100    8,505,730
Электроника   17,402,330   16,968,100
(5, 2)
Категория с максимальной выручкой Электроника   34,370,430 
Категория с минимальным ростом/падением Одежда            -7.2%
Дом               +2.6%
Книги             +7.1%
Красота           +2.6%
Одежда            -7.2%
Электроника        -2.5%


## [База] 3. Средний чек по категориям в октябре
Соберите вторую таблицу: **число** заказов по категориям и месяцам (сумма маски, а не сумма amount).
Средний чек = выручка / число заказов, одной операцией над двумя таблицами. У какой категории он самый высокий?

In [94]:
# 3
table = np.array([[amount[clean & (cat == c) & (month == m)].sum() for m in months]
                  for c in categories])
table_count = np.array([[(clean & (cat == c) & (month == m)).sum() for m in months]
                  for c in categories])

avg=table/table_count
avg_total=table.sum(axis=1)/table_count.sum(axis=1)
max_check_x=avg_total.argmax()

print(f"{'Категория':<15} {'2026-09':>12} {'2026-10':>12} {'Общий ср. чек':>15}")
print("-" * 56)
for c, row_m, tot in zip(categories, avg, avg_total):
    print(f"{c:<15} {row_m[0]:>12,.0f} {row_m[1]:>12,.0f} {tot:>15,.0f}")

print(f"\nСамый высокий средний чек у категории «{categories[max_check_x]}»: {avg_total[max_check_x]:,.0f}")
max_check_oct=avg[:, 1].argmax()
print(f"\nСамый высокий средний чек в октябре «{categories[max_check_oct]}»: {avg[:, 1][max_check_oct]:,.0f}")
print(max_check_oct)


Категория            2026-09      2026-10   Общий ср. чек
--------------------------------------------------------
Дом                   10,499       10,768          10,634
Книги                  4,020        3,816           3,912
Красота                6,187        6,159           6,173
Одежда                13,521       13,046          13,288
Электроника           32,773       33,600          33,176

Самый высокий средний чек у категории «Электроника»: 33,176

Самый высокий средний чек в октябре «Электроника»: 33,600
4


## [Стандарт] 4. Доля городов в выручке месяца
Используйте `table` городов с урока. Доля каждого города в выручке своего месяца: `table / table.sum(axis=0)`.
Проверьте, что каждый столбец в сумме даёт 1. Как изменилась доля Шымкента?

In [85]:
# 4
table = np.array([[amount[clean & (city == c) & (month == m)].sum() for m in months]
                  for c in cities])
ss=table / table.sum(axis=0)
print("Суммы столбцов по месяцам:", ss.sum(axis=0))

shym=np.where(cities=="Шымкент")[0]
# sept_ss = ss[shym, 0][0]
# oct_ss = ss[shym, 1][0]
# change = oct_ss - sept_ss
# print(f"\nДоля Шымкента:")
# print(f"- Сентябрь: {sept_ss:.2%}")
# print(f"- Октябрь:  {oct_ss:.2%}")
# print(f"- Изменение: {change:+.2%}")
x=ss[:, 1] - ss[:, 0]
print(f"\nДоля Шымкента:")
print(f"- Сентябрь: {ss[shym,0][0]:.2%}")
print(f"- Октябрь:  {ss[shym,1][0]:.2%}")
# print(f"- Изменение: {ss[shym,1][0]-ss[shym,0][0]:+.2%}")
print(f"- Изменение: {x[shym[0]]:+.2%}")
print("Получил выручки меньше чем окбтябре. Проверить по какой причине. Почему клиент стал покупать меньше чем в прошлом месяце. Может каких то товаров не хватает")


Суммы столбцов по месяцам: [1. 1.]

Доля Шымкента:
- Сентябрь: 12.93%
- Октябрь:  11.75%
- Изменение: -1.19%
- Изменение: -1.19%


## [Стандарт] 5. Кто дал падение
Вклад каждого города в общее падение выручки: изменение города / общее изменение. Сколько процентов падения даёт Шымкент?
Напишите менеджеру одну фразу: где искать причину.

In [100]:
# 5
unique_cities = np.unique(city)
city_table = np.array([[amount[(city == c) & (month == m)].sum() for m in months] for c in unique_cities])

city_diff = city_table[:, 1] - city_table[:, 0]
total_diff = city_diff.sum()
contribution = city_diff / total_diff
print(f"{'Город':<12} {'Изменение':>15} {'Вклад в динамику':>18}")
print("-" * 48)

for c, diff, contrib in zip(unique_cities, city_diff, contribution):
    print(f"{c:<12} {diff:>15,.0f} {contrib:>18.1%}")

shym_idx = list(unique_cities).index("Шымкент")

print(f"Шымкент даёт {contribution[shym_idx]:.1%} общего падения выручки.")


Город              Изменение   Вклад в динамику
------------------------------------------------
Актобе              -155,990               1.4%
Алматы           -10,713,650              93.0%
Астана              -750,000               6.5%
Атырау               -58,120               0.5%
Караганда           -339,300               2.9%
Шымкент              491,480              -4.3%
Шымкент даёт -4.3% общего падения выручки.


## [Стандарт] 6. Исправьте код от AI
AI написал код «доля сентября и октября внутри каждого города». Код падает. Объясните, почему, и исправьте.
```python
share = table / table.sum(axis=1)
```

In [104]:
# 6
# share = table / table.sum(axis=1)
# Ошибка из за несоотвествия матрицы размерность исходной (5,2) а он пытается (5,1)
# Решение использовать параметр keepdims
shares = table / table.sum(axis=1, keepdims=True)
print(shares)

[[0.49367512 0.50632488]
 [0.48286428 0.51713572]
 [0.49356893 0.50643107]
 [0.51871149 0.48128851]
 [0.50631691 0.49368309]]


## [Челлендж] 7. Стандартизация суммы заказа
Для чистых оплаченных заказов посчитайте z-оценку: `z = (x - x.mean()) / x.std()`.
Сколько заказов с |z| > 3 и какая это доля? Проверьте, что у `z` среднее ≈ 0 и std = 1.

In [ ]:
# 7

## [Челлендж] 8. Масштабирование признаков (заготовка для ML)
Соберите матрицу `X` из двух столбцов: `amount` и `days` для чистых заказов без пропусков в `days` (`np.column_stack`).
Приведите каждый столбец к диапазону [0, 1] одной строкой: `(X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))`.
Проверьте min и max каждого столбца. Объясните своими словами, зачем модели нужны признаки в одном масштабе.

In [ ]:
# 8